In [2]:

from google.colab import drive
from pathlib import Path
import hashlib
import json
import pandas as pd

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

# Load frozen research manifests.
with (BASE / "configs/holdout_partition_manifest_v1.json").open() as f:
    partition_manifest = json.load(f)

with (BASE / "outputs/llm_main_v1/full_annotation_audit_v1.json").open() as f:
    annotation_audit = json.load(f)

with (BASE / "outputs/s_slte_evidence_v1/evidence_manifest_v1.json").open() as f:
    evidence_manifest = json.load(f)

with (BASE / "outputs/s_slte_calibration_v1/frozen_correction_rule_v1.json").open() as f:
    frozen_rule = json.load(f)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def id_fingerprint(ids):
    payload = "\n".join(
        str(int(x)) for x in sorted(ids)
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def text_fingerprint(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values("record_id").itertuples(index=False):
        digest.update(str(int(row.record_id)).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(row.text).encode("utf-8"))
        digest.update(b"\0")

    return digest.hexdigest()

# Validate the original annotation audit.
assert annotation_audit["status"] == "FULL_CHECKPOINT_AUDIT_PASSED"
assert annotation_audit["total_records"] == 15621
assert annotation_audit["invalid_predictions"] == 0

# Restore the exact frozen active-training IDs.
training_spec = partition_manifest["partitions"]["training"]

training_ids = set(map(int, training_spec["record_ids"]))

assert len(training_ids) == training_spec["count"] == 13621
assert id_fingerprint(training_ids) == training_spec["id_sha256"]

# Load only record IDs and original text, never reference labels.
source_training = pd.read_csv(
    BASE / "data/splits/noisy_training.csv",
    usecols=["record_id", "text"]
)

assert source_training["record_id"].is_unique
assert source_training["text"].notna().all()
assert training_ids.issubset(
    set(source_training["record_id"].astype(int))
)

# Filter to the frozen 13,621 active training records.
training_text = (
    source_training.loc[
        source_training["record_id"].isin(training_ids)
    ]
    .sort_values("record_id")
    .reset_index(drop=True)
)

assert len(training_text) == 13621
assert set(training_text["record_id"].astype(int)) == training_ids
assert text_fingerprint(training_text) == (
    training_spec["raw_text_sha256"]
)

# Load and verify original LLM annotations.
annotation_path = (
    BASE / "outputs/llm_main_v1/training_annotations_v1.csv"
)

assert sha256_file(annotation_path) == (
    annotation_audit["exports"]["training"]["sha256"]
)

annotations = pd.read_csv(annotation_path)

assert len(annotations) == 13621
assert annotations["record_id"].is_unique
assert set(annotations["record_id"].astype(int)) == training_ids
assert annotations["llm_label"].isin(range(4)).all()

# Load and verify saved semantic neighborhood evidence.
evidence_path = (
    BASE / "outputs/s_slte_evidence_v1/training_neighbor_evidence_v1.csv"
)

assert sha256_file(evidence_path) == (
    evidence_manifest["partitions"]["training"]["sha256"]
)

evidence = pd.read_csv(evidence_path)

assert len(evidence) == 13621
assert evidence["record_id"].is_unique
assert set(evidence["record_id"].astype(int)) == training_ids
assert evidence["has_positive_similarity"].all()
assert evidence["neighbor_top_label"].isin(range(4)).all()

for class_id in range(4):
    assert f"support_class_{class_id}" in evidence.columns

# Verify alignment between LLM labels and semantic evidence.
aligned = annotations[["record_id", "llm_label"]].merge(
    evidence[["record_id", "llm_label"]],
    on="record_id",
    validate="one_to_one",
    suffixes=("_annotation", "_evidence")
)

assert (
    aligned["llm_label_annotation"]
    == aligned["llm_label_evidence"]
).all()

# Verify the previously frozen correction rule.
assert frozen_rule["method"] == (
    "S_SLTE_SEMANTIC_ONLY_CONSERVATIVE_RULE"
)
assert frozen_rule["min_neighbor_support"] == 0.80
assert frozen_rule["min_semantic_support_gap"] == 0.50
assert frozen_rule["k"] == 20
assert frozen_rule["training_labels_modified"] is False

# Create a text-and-LLM-label dataset for the next step.
training_df = training_text.merge(
    annotations[["record_id", "llm_label"]],
    on="record_id",
    validate="one_to_one"
)

assert len(training_df) == 13621
assert "label" not in training_df.columns

print("STAGE 13 — INPUT RESTORATION")

print("\nSOURCE DATA")
print("Original source rows:", len(source_training))
print("Frozen active training rows:", len(training_text))
print("Excluded source rows:", len(source_training) - len(training_text))

print("\nVERIFICATION")
print("Frozen ID fingerprint: VERIFIED")
print("Frozen text fingerprint: VERIFIED")
print("LLM annotations: VERIFIED")
print("Semantic evidence: VERIFIED")
print("Record ID alignment: VERIFIED")
print("Original annotation audit: VERIFIED")

print("\nFROZEN S-SLTE RULE")
print("Minimum support:", frozen_rule["min_neighbor_support"])
print("Minimum gap:", frozen_rule["min_semantic_support_gap"])
print("Clean anchor K:", frozen_rule["k"])

print("\nRESEARCH INTEGRITY")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("Training labels modified: NO")
print("New inference performed: NO")

print("\nSTATUS: STAGE 13 INPUT VERIFICATION PASSED")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STAGE 13 — INPUT RESTORATION

SOURCE DATA
Original source rows: 16200
Frozen active training rows: 13621
Excluded source rows: 2579

VERIFICATION
Frozen ID fingerprint: VERIFIED
Frozen text fingerprint: VERIFIED
LLM annotations: VERIFIED
Semantic evidence: VERIFIED
Record ID alignment: VERIFIED
Original annotation audit: VERIFIED

FROZEN S-SLTE RULE
Minimum support: 0.8
Minimum gap: 0.5
Clean anchor K: 20

RESEARCH INTEGRITY
Training reference labels accessed: NO
Holdout reference labels accessed: NO
Training labels modified: NO
New inference performed: NO

STATUS: STAGE 13 INPUT VERIFICATION PASSED


In [3]:

import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

# Stage 13 — Frozen S-SLTE training correction.
OUTPUT_DIR = BASE / "outputs/s_slte_training_correction_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RULE_PATH = (
    BASE / "outputs/s_slte_calibration_v1"
    / "frozen_correction_rule_v1.json"
)

ANNOTATION_PATH = (
    BASE / "outputs/llm_main_v1/training_annotations_v1.csv"
)

EVIDENCE_PATH = (
    BASE / "outputs/s_slte_evidence_v1"
    / "training_neighbor_evidence_v1.csv"
)

assert len(training_df) == 13621
assert len(evidence) == 13621
assert training_df["record_id"].is_unique
assert evidence["record_id"].is_unique
assert "label" not in training_df.columns
assert "label" not in evidence.columns

assert frozen_rule["min_neighbor_support"] == 0.80
assert frozen_rule["min_semantic_support_gap"] == 0.50
assert frozen_rule["k"] == 20

assert sha256_file(ANNOTATION_PATH) == (
    annotation_audit["exports"]["training"]["sha256"]
)

assert sha256_file(EVIDENCE_PATH) == (
    evidence_manifest["partitions"]["training"]["sha256"]
)

with RULE_PATH.open(encoding="utf-8") as f:
    saved_rule = json.load(f)

assert saved_rule == frozen_rule

SUPPORT_COLUMNS = [
    f"support_class_{class_id}"
    for class_id in range(4)
]

# Join existing LLM labels and anchor-neighborhood evidence.
combined = training_df[
    ["record_id", "llm_label"]
].merge(
    evidence[
        [
            "record_id",
            "llm_label",
            "neighbor_top_label",
            "neighbor_top_support",
            "llm_label_support",
            "has_positive_similarity",
            *SUPPORT_COLUMNS
        ]
    ],
    on=["record_id", "llm_label"],
    how="inner",
    validate="one_to_one"
).sort_values("record_id").reset_index(drop=True)

assert len(combined) == 13621
assert set(combined["record_id"].astype(int)) == training_ids
assert combined["has_positive_similarity"].all()

llm = combined["llm_label"].to_numpy(dtype=int)

neighbor = combined[
    "neighbor_top_label"
].to_numpy(dtype=int)

support = combined[
    SUPPORT_COLUMNS
].to_numpy(dtype=np.float64)

assert np.isin(llm, range(4)).all()
assert np.isin(neighbor, range(4)).all()
assert np.isfinite(support).all()
assert np.all(support >= 0)
assert np.allclose(
    support.sum(axis=1), 1.0, atol=1e-6
)
assert np.array_equal(
    support.argmax(axis=1), neighbor
)

top_support = support.max(axis=1)
llm_support = support[np.arange(len(llm)), llm]

assert np.allclose(
    top_support,
    combined["neighbor_top_support"].to_numpy(),
    atol=1e-6
)

assert np.allclose(
    llm_support,
    combined["llm_label_support"].to_numpy(),
    atol=1e-6
)

support_gap = np.maximum(
    top_support - llm_support, 0.0
)

conflict = llm != neighbor

assert int(conflict.sum()) == 2590

# Apply the previously frozen rule without retuning.
should_correct = (
    conflict
    & (
        top_support >=
        frozen_rule["min_neighbor_support"]
    )
    & (
        support_gap >=
        frozen_rule["min_semantic_support_gap"]
    )
)

corrected = np.where(
    should_correct,
    neighbor,
    llm
)

assert np.array_equal(
    corrected[~should_correct],
    llm[~should_correct]
)

assert np.array_equal(
    corrected[should_correct],
    neighbor[should_correct]
)

assert np.isin(corrected, range(4)).all()

# Preserve decisions separately from the original annotations.
decisions = pd.DataFrame({
    "record_id": combined["record_id"].astype("int64"),
    "original_llm_label": llm,
    "neighbor_proposed_label": neighbor,
    "neighbor_top_support": top_support,
    "llm_label_support": llm_support,
    "semantic_support_gap": support_gap,
    "llm_neighbor_conflict": conflict,
    "correction_applied": should_correct,
    "corrected_label": corrected
})

corrected_labels = decisions[
    [
        "record_id",
        "original_llm_label",
        "corrected_label",
        "correction_applied"
    ]
].copy()

assert len(decisions) == len(corrected_labels) == 13621
assert decisions["record_id"].is_unique
assert corrected_labels["record_id"].is_unique

def save_immutable(path, content):
    payload = content.encode("utf-8")

    if path.exists():
        assert path.read_bytes() == payload, (
            f"Existing artifact differs: {path}"
        )
        return "EXISTING_VERIFIED"

    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(payload)
    os.replace(temporary, path)

    assert path.read_bytes() == payload
    return "NEW_SAVED"

decisions_path = (
    OUTPUT_DIR / "training_correction_decisions_v1.csv"
)

labels_path = (
    OUTPUT_DIR / "training_corrected_labels_v1.csv"
)

decisions_status = save_immutable(
    decisions_path,
    decisions.to_csv(
        index=False,
        float_format="%.8f",
        lineterminator="\n"
    )
)

labels_status = save_immutable(
    labels_path,
    corrected_labels.to_csv(
        index=False,
        lineterminator="\n"
    )
)

correction_count = int(should_correct.sum())

class_names = [
    "World", "Sports", "Business", "Sci/Tech"
]

class_summary = []

for class_id, class_name in enumerate(class_names):
    class_summary.append({
        "class_name": class_name,
        "original_count": int((llm == class_id).sum()),
        "corrected_count": int((corrected == class_id).sum()),
        "incoming_corrections": int(
            (should_correct & (neighbor == class_id)).sum()
        ),
        "outgoing_corrections": int(
            (should_correct & (llm == class_id)).sum()
        )
    })

manifest = {
    "protocol_version": "v1",
    "stage": "FROZEN_TRAINING_LABEL_CORRECTION",
    "method": frozen_rule["method"],
    "total_records": 13621,
    "llm_neighbor_conflicts": int(conflict.sum()),
    "corrections_applied": correction_count,
    "original_labels_retained": int((~should_correct).sum()),
    "abstained_conflicts": int(
        (conflict & ~should_correct).sum()
    ),
    "min_support": 0.80,
    "min_gap": 0.50,
    "k": 20,
    "class_summary": class_summary,
    "source_sha256": {
        "frozen_rule": sha256_file(RULE_PATH),
        "llm_annotations": sha256_file(ANNOTATION_PATH),
        "semantic_evidence": sha256_file(EVIDENCE_PATH)
    },
    "output_sha256": {
        "decisions": sha256_file(decisions_path),
        "corrected_labels": sha256_file(labels_path)
    },
    "training_reference_labels_accessed": False,
    "holdout_reference_labels_accessed": False,
    "original_llm_annotations_modified": False,
    "downstream_classifier_trained": False,
    "correction_accuracy_measured": False
}

manifest_path = (
    OUTPUT_DIR / "training_correction_manifest_v1.json"
)

manifest_status = save_immutable(
    manifest_path,
    json.dumps(manifest, indent=2) + "\n"
)

print("STAGE 13 — TRAINING CORRECTION COMPLETED")

print("\nCORRECTION SUMMARY")
print("Training records:", len(corrected_labels))
print("LLM–Neighbor conflicts:", int(conflict.sum()))
print("Corrections applied:", correction_count)
print("Original labels retained:", int((~should_correct).sum()))
print(
    "Conflicts abstained:",
    int((conflict & ~should_correct).sum())
)
print(
    "Correction coverage:",
    f"{100 * correction_count / len(corrected_labels):.2f}%"
)

print("\nCLASS DISTRIBUTION")
display(pd.DataFrame(class_summary))

print("\nSAVED ARTIFACTS")
print("Decisions:", decisions_status)
print("Corrected labels:", labels_status)
print("Manifest:", manifest_status)
print("Directory:", OUTPUT_DIR)

print("\nRESEARCH INTEGRITY")
print("Frozen rule unchanged: YES")
print("Source fingerprints verified: YES")
print("Original annotation file modified: NO")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("Downstream model trained: NO")
print("Correction accuracy measured: NO")

print("\nSTATUS: STAGE 13 TRAINING CORRECTION PASSED")


STAGE 13 — TRAINING CORRECTION COMPLETED

CORRECTION SUMMARY
Training records: 13621
LLM–Neighbor conflicts: 2590
Corrections applied: 705
Original labels retained: 12916
Conflicts abstained: 1885
Correction coverage: 5.18%

CLASS DISTRIBUTION


,class_name,original_count,corrected_count,incoming_corrections,outgoing_corrections
0,World,4199,4008,10,201
1,Sports,3532,3542,49,39
2,Business,4015,3662,96,449
3,Sci/Tech,1875,2409,550,16



SAVED ARTIFACTS
Decisions: NEW_SAVED
Corrected labels: NEW_SAVED
Manifest: NEW_SAVED
Directory: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_training_correction_v1

RESEARCH INTEGRITY
Frozen rule unchanged: YES
Source fingerprints verified: YES
Original annotation file modified: NO
Training reference labels accessed: NO
Holdout reference labels accessed: NO
Downstream model trained: NO
Correction accuracy measured: NO

STATUS: STAGE 13 TRAINING CORRECTION PASSED


In [4]:

import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

# Stage 13 — Prepare matched downstream training datasets.
DOWNSTREAM_DIR = BASE / "outputs/s_slte_downstream_v1"
DOWNSTREAM_DIR.mkdir(parents=True, exist_ok=True)

CORRECTION_DIR = BASE / "outputs/s_slte_training_correction_v1"

LABELS_PATH = (
    CORRECTION_DIR / "training_corrected_labels_v1.csv"
)

DECISIONS_PATH = (
    CORRECTION_DIR / "training_correction_decisions_v1.csv"
)

CORRECTION_MANIFEST_PATH = (
    CORRECTION_DIR / "training_correction_manifest_v1.json"
)

RULE_PATH = (
    BASE / "outputs/s_slte_calibration_v1"
    / "frozen_correction_rule_v1.json"
)

def file_digest(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def save_immutable(path, content):
    payload = content.encode("utf-8")

    if path.exists():
        assert path.read_bytes() == payload, (
            f"Existing artifact differs: {path}"
        )
        return "EXISTING_VERIFIED"

    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_bytes(payload)
    os.replace(temp, path)

    assert path.read_bytes() == payload
    return "NEW_SAVED"

# Verify saved correction artifacts.
with CORRECTION_MANIFEST_PATH.open(encoding="utf-8") as f:
    correction_manifest = json.load(f)

assert correction_manifest["total_records"] == 13621
assert correction_manifest["corrections_applied"] == 705

assert file_digest(LABELS_PATH) == (
    correction_manifest["output_sha256"]["corrected_labels"]
)

assert file_digest(DECISIONS_PATH) == (
    correction_manifest["output_sha256"]["decisions"]
)

assert file_digest(RULE_PATH) == (
    correction_manifest["source_sha256"]["frozen_rule"]
)

# Load the two saved correction outputs.
corrected_labels = pd.read_csv(LABELS_PATH)
saved_decisions = pd.read_csv(DECISIONS_PATH)

assert len(corrected_labels) == 13621
assert len(saved_decisions) == 13621
assert corrected_labels["record_id"].is_unique
assert saved_decisions["record_id"].is_unique

# Verify that both saved artifacts describe identical decisions.
decision_check = corrected_labels.merge(
    saved_decisions[
        [
            "record_id",
            "original_llm_label",
            "corrected_label",
            "correction_applied"
        ]
    ],
    on="record_id",
    validate="one_to_one",
    suffixes=("_labels", "_decisions")
)

for column in [
    "original_llm_label",
    "corrected_label",
    "correction_applied"
]:
    assert (
        decision_check[f"{column}_labels"]
        == decision_check[f"{column}_decisions"]
    ).all()

# Restore frozen original text and LLM labels.
source = (
    training_df[["record_id", "text", "llm_label"]]
    .sort_values("record_id")
    .reset_index(drop=True)
)

assert len(source) == 13621
assert source["record_id"].is_unique
assert source["text"].notna().all()

assert text_fingerprint(
    source[["record_id", "text"]]
) == training_spec["raw_text_sha256"]

combined = source.merge(
    corrected_labels,
    on="record_id",
    validate="one_to_one"
).sort_values("record_id").reset_index(drop=True)

assert len(combined) == 13621
assert set(combined["record_id"].astype(int)) == training_ids

assert (
    combined["llm_label"]
    == combined["original_llm_label"]
).all()

original = combined["llm_label"].to_numpy(dtype=int)
corrected = combined["corrected_label"].to_numpy(dtype=int)
changed = combined["correction_applied"].to_numpy(dtype=bool)

assert np.isin(original, range(4)).all()
assert np.isin(corrected, range(4)).all()
assert np.array_equal(changed, original != corrected)
assert int(changed.sum()) == 705

# Independently reconstruct decisions from frozen neighborhood evidence.
evidence_check = combined[
    ["record_id", "llm_label"]
].merge(
    evidence[
        ["record_id", "neighbor_top_label"]
        + [f"support_class_{i}" for i in range(4)]
    ],
    on="record_id",
    validate="one_to_one"
).sort_values("record_id").reset_index(drop=True)

support = evidence_check[
    [f"support_class_{i}" for i in range(4)]
].to_numpy(dtype=float)

neighbor = evidence_check[
    "neighbor_top_label"
].to_numpy(dtype=int)

assert np.isfinite(support).all()
assert np.allclose(support.sum(axis=1), 1.0, atol=1e-6)
assert np.array_equal(support.argmax(axis=1), neighbor)

top_support = support.max(axis=1)
llm_support = support[np.arange(len(original)), original]

expected_changed = (
    (original != neighbor)
    & (top_support >= frozen_rule["min_neighbor_support"])
    & (
        top_support - llm_support
        >= frozen_rule["min_semantic_support_gap"]
    )
)

assert np.array_equal(changed, expected_changed)
assert np.array_equal(
    corrected,
    np.where(expected_changed, neighbor, original)
)

# Prepare matched datasets using generated training targets.
baseline_df = combined[
    ["record_id", "text"]
].copy()

baseline_df["label"] = original

corrected_df = combined[
    ["record_id", "text"]
].copy()

corrected_df["label"] = corrected

assert len(baseline_df) == len(corrected_df) == 13621

assert baseline_df["record_id"].equals(
    corrected_df["record_id"]
)

assert baseline_df["text"].equals(
    corrected_df["text"]
)

assert int(
    (baseline_df["label"] != corrected_df["label"]).sum()
) == 705

# Save the two datasets without modifying source artifacts.
BASELINE_PATH = (
    DOWNSTREAM_DIR / "baseline_training_v1.csv"
)

CORRECTED_PATH = (
    DOWNSTREAM_DIR / "corrected_training_v1.csv"
)

baseline_status = save_immutable(
    BASELINE_PATH,
    baseline_df.to_csv(
        index=False,
        lineterminator="\n"
    )
)

corrected_status = save_immutable(
    CORRECTED_PATH,
    corrected_df.to_csv(
        index=False,
        lineterminator="\n"
    )
)

# Record precise provenance for reproducibility.
dataset_manifest = {
    "protocol_version": "v1",
    "experiment": "MATCHED_DOWNSTREAM_TRAINING_DATA",
    "records_per_condition": 13621,
    "conditions": [
        "ORIGINAL_LLM_LABELS",
        "FROZEN_S_SLTE_CORRECTED_LABELS"
    ],
    "text_policy": "IDENTICAL_FROZEN_ORIGINAL_TEXT",
    "record_order": "ASCENDING_RECORD_ID",
    "training_target_semantics": (
        "Generated labels; not trusted reference labels"
    ),
    "label_changes": 705,
    "class_count_original": {
        str(i): int((original == i).sum())
        for i in range(4)
    },
    "class_count_corrected": {
        str(i): int((corrected == i).sum())
        for i in range(4)
    },
    "source_sha256": {
        "partition_manifest": file_digest(
            BASE / "configs/holdout_partition_manifest_v1.json"
        ),
        "correction_manifest": file_digest(
            CORRECTION_MANIFEST_PATH
        ),
        "frozen_rule": file_digest(RULE_PATH)
    },
    "output_sha256": {
        "baseline_training": file_digest(BASELINE_PATH),
        "corrected_training": file_digest(CORRECTED_PATH)
    },
    "reference_labels_accessed": False,
    "holdout_accessed": False,
    "downstream_training_started": False
}

DATASET_MANIFEST_PATH = (
    DOWNSTREAM_DIR / "matched_training_manifest_v1.json"
)

manifest_status = save_immutable(
    DATASET_MANIFEST_PATH,
    json.dumps(dataset_manifest, indent=2) + "\n"
)

print("STAGE 13 — MATCHED TRAINING DATASETS")

print("\nDATASET VERIFICATION")
print("Baseline records:", len(baseline_df))
print("Corrected records:", len(corrected_df))
print("Identical record IDs: YES")
print("Identical original texts: YES")
print("Changed labels:", int(changed.sum()))

print("\nTRAINING CLASS DISTRIBUTION")
display(pd.DataFrame([
    {
        "class": name,
        "baseline": int((original == i).sum()),
        "corrected": int((corrected == i).sum())
    }
    for i, name in enumerate(
        ["World", "Sports", "Business", "Sci/Tech"]
    )
]))

print("\nARTIFACTS")
print("Baseline CSV:", baseline_status)
print("Corrected CSV:", corrected_status)
print("Manifest:", manifest_status)
print("Directory:", DOWNSTREAM_DIR)

print("\nRESEARCH INTEGRITY")
print("Frozen rule verified: YES")
print("Correction decisions independently reproduced: YES")
print("Training reference labels accessed: NO")
print("Holdout accessed: NO")
print("Model training started: NO")

print("\nSTATUS: MATCHED TRAINING DATASETS VERIFIED")


STAGE 13 — MATCHED TRAINING DATASETS

DATASET VERIFICATION
Baseline records: 13621
Corrected records: 13621
Identical record IDs: YES
Identical original texts: YES
Changed labels: 705

TRAINING CLASS DISTRIBUTION


,class,baseline,corrected
0,World,4199,4008
1,Sports,3532,3542
2,Business,4015,3662
3,Sci/Tech,1875,2409



ARTIFACTS
Baseline CSV: NEW_SAVED
Corrected CSV: NEW_SAVED
Manifest: NEW_SAVED
Directory: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1

RESEARCH INTEGRITY
Frozen rule verified: YES
Correction decisions independently reproduced: YES
Training reference labels accessed: NO
Holdout accessed: NO
Model training started: NO

STATUS: MATCHED TRAINING DATASETS VERIFIED


In [6]:

%pip install -q "huggingface_hub==1.33.0"

import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from huggingface_hub import HfApi

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
DOWNSTREAM_DIR = BASE / "outputs/s_slte_downstream_v1"

BASELINE_PATH = DOWNSTREAM_DIR / "baseline_training_v1.csv"
CORRECTED_PATH = DOWNSTREAM_DIR / "corrected_training_v1.csv"
DATASET_MANIFEST_PATH = (
    DOWNSTREAM_DIR / "matched_training_manifest_v1.json"
)
PROTOCOL_PATH = (
    DOWNSTREAM_DIR / "downstream_training_protocol_v1.json"
)

MODEL_ID = "distilbert/distilbert-base-uncased"
SEEDS = [42, 123, 2026]

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def text_fingerprint(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values("record_id").itertuples(index=False):
        digest.update(str(int(row.record_id)).encode())
        digest.update(b"\0")
        digest.update(str(row.text).encode())
        digest.update(b"\0")

    return digest.hexdigest()

# Verify the frozen training datasets.
with DATASET_MANIFEST_PATH.open(encoding="utf-8") as f:
    dataset_manifest = json.load(f)

assert dataset_manifest["records_per_condition"] == 13621
assert dataset_manifest["label_changes"] == 705

assert file_hash(BASELINE_PATH) == (
    dataset_manifest["output_sha256"]["baseline_training"]
)
assert file_hash(CORRECTED_PATH) == (
    dataset_manifest["output_sha256"]["corrected_training"]
)

baseline = pd.read_csv(BASELINE_PATH)
corrected = pd.read_csv(CORRECTED_PATH)

assert len(baseline) == len(corrected) == 13621
assert baseline["record_id"].is_unique
assert corrected["record_id"].is_unique

assert baseline["record_id"].equals(corrected["record_id"])
assert baseline["text"].equals(corrected["text"])

assert baseline["label"].isin(range(4)).all()
assert corrected["label"].isin(range(4)).all()

changes = int(
    (baseline["label"] != corrected["label"]).sum()
)
assert changes == 705

# Match the exact frozen active-training text.
with (
    BASE / "configs/holdout_partition_manifest_v1.json"
).open(encoding="utf-8") as f:
    partition_manifest = json.load(f)

training_spec = partition_manifest["partitions"]["training"]

assert text_fingerprint(
    baseline[["record_id", "text"]]
) == training_spec["raw_text_sha256"]

# Pin the pretrained model revision.
if PROTOCOL_PATH.exists():
    with PROTOCOL_PATH.open(encoding="utf-8") as f:
        previous_protocol = json.load(f)

    model_revision = previous_protocol["model_revision_sha"]
else:
    model_revision = HfApi().model_info(MODEL_ID).sha

assert isinstance(model_revision, str)
assert len(model_revision) == 40
assert all(c in "0123456789abcdef" for c in model_revision)

protocol = {
    "protocol_version": "v1",
    "experiment": "S_SLTE_MATCHED_DOWNSTREAM_COMPARISON",
    "model_id": MODEL_ID,
    "model_revision_sha": model_revision,
    "model_task": "SEQUENCE_CLASSIFICATION",
    "num_labels": 4,
    "class_names": [
        "World", "Sports", "Business", "Sci/Tech"
    ],
    "conditions": {
        "baseline": BASELINE_PATH.name,
        "corrected": CORRECTED_PATH.name
    },
    "seeds": SEEDS,
    "training": {
        "epochs": 2,
        "train_batch_size": 16,
        "gradient_accumulation_steps": 2,
        "learning_rate": 2e-5,
        "optimizer": "AdamW",
        "weight_decay": 0.01,
        "warmup_ratio": 0.06,
        "lr_scheduler": "linear",
        "max_length": 256,
        "dynamic_padding": True,
        "drop_last": False,
        "class_weights": None,
        "early_stopping": False,
        "load_best_model_at_end": False,
        "final_checkpoint_policy": "LAST_EPOCH"
    },
    "matched_controls": {
        "same_pretrained_revision": True,
        "same_training_record_ids": True,
        "same_training_text": True,
        "same_record_order": True,
        "same_seed_per_pair": True,
        "reset_seed_before_model_initialization": True,
        "same_batch_order_per_pair": True,
        "same_optimizer_and_schedule": True,
        "only_training_labels_differ": True
    },
    "evaluation_plan": {
        "primary_metric": "holdout_macro_f1",
        "secondary_metrics": [
            "holdout_accuracy",
            "per_class_precision_recall_f1"
        ],
        "report_per_seed": True,
        "report_mean_and_standard_deviation": True,
        "paired_comparison": True,
        "holdout_evaluation_timing": (
            "AFTER_ALL_SIX_TRAINING_RUNS"
        ),
        "holdout_used_for_tuning": False
    },
    "input_sha256": {
        "baseline_csv": file_hash(BASELINE_PATH),
        "corrected_csv": file_hash(CORRECTED_PATH),
        "matched_dataset_manifest": file_hash(
            DATASET_MANIFEST_PATH
        )
    },
    "training_reference_labels_accessed": False,
    "holdout_reference_labels_accessed": False,
    "training_started": False,
    "evaluation_started": False
}

content = json.dumps(protocol, indent=2) + "\n"

if PROTOCOL_PATH.exists():
    assert PROTOCOL_PATH.read_text(
        encoding="utf-8"
    ) == content
    status = "EXISTING_VERIFIED"
else:
    temporary = PROTOCOL_PATH.with_suffix(".json.tmp")
    temporary.write_text(content, encoding="utf-8")
    os.replace(temporary, PROTOCOL_PATH)
    status = "NEW_SAVED"

print("STAGE 13 — DOWNSTREAM TRAINING PROTOCOL")
print("Model:", MODEL_ID)
print("Model revision:", model_revision)
print("Training records per condition:", len(baseline))
print("Changed labels:", changes)
print("Random seeds:", SEEDS)
print("Epochs:", protocol["training"]["epochs"])
print("Total planned training runs:", 2 * len(SEEDS))
print("Learning rate:", protocol["training"]["learning_rate"])
print("Maximum length:", protocol["training"]["max_length"])

print("\nREPRODUCIBILITY")
print("Training text fingerprint: VERIFIED")
print("Training dataset fingerprints: VERIFIED")
print("Model revision: PINNED")
print("Protocol:", status)
print("Saved path:", PROTOCOL_PATH)

print("\nRESEARCH INTEGRITY")
print("Training labels are generated targets: YES")
print("Training reference labels accessed: NO")
print("Holdout reference labels accessed: NO")
print("Training started: NO")
print("Evaluation started: NO")

print("\nSTATUS: DOWNSTREAM TRAINING PROTOCOL FROZEN")


STAGE 13 — DOWNSTREAM TRAINING PROTOCOL
Model: distilbert/distilbert-base-uncased
Model revision: 12040accade4e8a0f71eabdb258fecc2e7e948be
Training records per condition: 13621
Changed labels: 705
Random seeds: [42, 123, 2026]
Epochs: 2
Total planned training runs: 6
Learning rate: 2e-05
Maximum length: 256

REPRODUCIBILITY
Training text fingerprint: VERIFIED
Training dataset fingerprints: VERIFIED
Model revision: PINNED
Protocol: EXISTING_VERIFIED
Saved path: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/downstream_training_protocol_v1.json

RESEARCH INTEGRITY
Training labels are generated targets: YES
Training reference labels accessed: NO
Holdout reference labels accessed: NO
Training started: NO
Evaluation started: NO

STATUS: DOWNSTREAM TRAINING PROTOCOL FROZEN


In [2]:

from google.colab import drive
from pathlib import Path
import json

# Restore Google Drive after a runtime change.
drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

PROTOCOL_PATH = (
    BASE / "outputs/s_slte_downstream_v1"
    / "downstream_training_protocol_v1.json"
)

print("Drive mounted:", Path("/content/drive/MyDrive").is_dir())
print("Research workspace exists:", BASE.is_dir())
print("Protocol exists:", PROTOCOL_PATH.is_file())

if PROTOCOL_PATH.is_file():
    with PROTOCOL_PATH.open(encoding="utf-8") as f:
        protocol = json.load(f)

    assert protocol["model_id"] == (
        "distilbert/distilbert-base-uncased"
    )
    assert protocol["training"]["epochs"] == 2
    assert protocol["seeds"] == [42, 123, 2026]

    print("\nModel:", protocol["model_id"])
    print("Revision:", protocol["model_revision_sha"])
    print("Training runs:", 2 * len(protocol["seeds"]))
    print("\nSTATUS: TRAINING PROTOCOL RESTORED")

else:
    print("\nProtocol not found at:", PROTOCOL_PATH)

    if BASE.is_dir():
        output_dir = BASE / "outputs"
        print(
            "Downstream folder exists:",
            (output_dir / "s_slte_downstream_v1").is_dir()
        )
    else:
        print("Check the mounted Google account and workspace path.")

    print("\nSTATUS: DRIVE PATH NEEDS INVESTIGATION")


Mounted at /content/drive
Drive mounted: True
Research workspace exists: True
Protocol exists: True

Model: distilbert/distilbert-base-uncased
Revision: 12040accade4e8a0f71eabdb258fecc2e7e948be
Training runs: 6

STATUS: TRAINING PROTOCOL RESTORED


In [3]:

%pip install -q "transformers==5.20.1" "accelerate" "safetensors"

import os
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import json
import random
import hashlib
import importlib.metadata as metadata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
DOWNSTREAM_DIR = BASE / "outputs/s_slte_downstream_v1"

PROTOCOL_PATH = (
    DOWNSTREAM_DIR / "downstream_training_protocol_v1.json"
)

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

assert PROTOCOL_PATH.is_file()

with PROTOCOL_PATH.open(encoding="utf-8") as f:
    protocol = json.load(f)

MODEL_ID = protocol["model_id"]
MODEL_REVISION = protocol["model_revision_sha"]

assert MODEL_ID == "distilbert/distilbert-base-uncased"
assert MODEL_REVISION == (
    "12040accade4e8a0f71eabdb258fecc2e7e948be"
)
assert protocol["num_labels"] == 4
assert protocol["seeds"] == [42, 123, 2026]
assert protocol["training"]["epochs"] == 2
assert transformers.__version__ == "5.20.1"

assert torch.cuda.is_available(), "Enable T4 GPU first."

DEVICE = torch.device("cuda:0")

# Reproducible smoke-test initialization.
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Load exact pinned public checkpoint.
downstream_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    token=False,
    use_fast=True
)

downstream_model = (
    AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=False,
        num_labels=4
    )
)

downstream_model.to(DEVICE)
downstream_model.eval()

assert downstream_model.config.num_labels == 4
assert downstream_tokenizer.is_fast
assert next(downstream_model.parameters()).device.type == "cuda"

# Verify both frozen training datasets.
BASELINE_PATH = DOWNSTREAM_DIR / "baseline_training_v1.csv"
CORRECTED_PATH = DOWNSTREAM_DIR / "corrected_training_v1.csv"

assert file_hash(BASELINE_PATH) == (
    protocol["input_sha256"]["baseline_csv"]
)

assert file_hash(CORRECTED_PATH) == (
    protocol["input_sha256"]["corrected_csv"]
)

sample = pd.read_csv(BASELINE_PATH, nrows=4)

assert len(sample) == 4
assert sample["text"].notna().all()

inputs = downstream_tokenizer(
    sample["text"].astype(str).tolist(),
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

assert inputs["input_ids"].shape[0] == 4
assert inputs["input_ids"].shape[1] <= 256

inputs = {
    key: value.to(DEVICE)
    for key, value in inputs.items()
}

# Forward pass only; no optimizer updates.
with torch.inference_mode():
    logits = downstream_model(**inputs).logits

assert logits.shape == (4, 4)
assert torch.isfinite(logits).all()

print("STAGE 13 — DISTILBERT READINESS TEST")

print("\nMODEL")
print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("Accelerate:", metadata.version("accelerate"))

print("\nGPU")
print("Device:", torch.cuda.get_device_name(0))
print(
    "Allocated VRAM:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print("\nVALIDATION")
print("Classes:", downstream_model.config.num_labels)
print("Tokenizer:", type(downstream_tokenizer).__name__)
print("Forward-pass shape:", tuple(logits.shape))
print("Finite logits: YES")
print("Training datasets: VERIFIED")
print("Pinned checkpoint: VERIFIED")

print("\nRESEARCH INTEGRITY")
print("Training performed: NO")
print("Optimizer updates: NO")
print("Training reference labels accessed: NO")
print("Holdout accessed: NO")

print("\nSTATUS: DOWNSTREAM MODEL READY FOR TRAINING")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


STAGE 13 — DISTILBERT READINESS TEST

MODEL
Model: distilbert/distilbert-base-uncased
Revision: 12040accade4e8a0f71eabdb258fecc2e7e948be
Transformers: 5.20.1
PyTorch: 2.14.1+cu130
Accelerate: 1.15.0

GPU
Device: Tesla T4
Allocated VRAM: 0.26 GB

VALIDATION
Classes: 4
Tokenizer: BertTokenizer
Forward-pass shape: (4, 4)
Finite logits: YES
Training datasets: VERIFIED
Pinned checkpoint: VERIFIED

RESEARCH INTEGRITY
Training performed: NO
Optimizer updates: NO
Training reference labels accessed: NO
Holdout accessed: NO

STATUS: DOWNSTREAM MODEL READY FOR TRAINING


In [4]:

import os
import gc
import json
import math
import random
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
ROOT = BASE / "outputs/s_slte_downstream_v1"

with (ROOT / "downstream_training_protocol_v1.json").open(
    encoding="utf-8"
) as f:
    cfg = json.load(f)

SEED = 42
CONDITION = "baseline"

assert SEED in cfg["seeds"]
assert cfg["model_id"] == "distilbert/distilbert-base-uncased"
assert cfg["training"]["epochs"] == 2
assert cfg["training"]["train_batch_size"] == 16
assert cfg["training"]["gradient_accumulation_steps"] == 2
assert cfg["training"]["final_checkpoint_policy"] == "LAST_EPOCH"
assert cfg["training"]["early_stopping"] is False
assert cfg["training"]["drop_last"] is False
assert torch.cuda.is_available()

DEVICE = torch.device("cuda:0")

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

protocol_path = ROOT / "downstream_training_protocol_v1.json"
data_path = ROOT / "baseline_training_v1.csv"
paired_path = ROOT / "corrected_training_v1.csv"

assert sha256(data_path) == cfg["input_sha256"]["baseline_csv"]
assert sha256(paired_path) == cfg["input_sha256"]["corrected_csv"]

data = pd.read_csv(data_path)
paired = pd.read_csv(paired_path)

assert len(data) == len(paired) == 13621
assert data["record_id"].is_unique
assert data["record_id"].equals(paired["record_id"])
assert data["text"].equals(paired["text"])
assert data["text"].notna().all()
assert data["label"].isin(range(4)).all()
assert int((data["label"] != paired["label"]).sum()) == 705

texts = data["text"].astype(str).tolist()
labels = data["label"].to_numpy(dtype=np.int64)

batch_size = cfg["training"]["train_batch_size"]
accumulation = cfg["training"]["gradient_accumulation_steps"]
epochs = cfg["training"]["epochs"]

batches_per_epoch = math.ceil(len(data) / batch_size)
updates_per_epoch = math.ceil(batches_per_epoch / accumulation)
total_updates = updates_per_epoch * epochs

warmup_steps = math.ceil(
    total_updates * cfg["training"]["warmup_ratio"]
)

RUN_DIR = ROOT / "training_runs" / f"{CONDITION}_seed_{SEED}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = RUN_DIR / "training_checkpoint.pt"
model_dir = RUN_DIR / "final_model"
summary_path = RUN_DIR / "training_summary_v1.json"

if summary_path.exists():
    with summary_path.open(encoding="utf-8") as f:
        completed = json.load(f)

    assert completed["status"] == "TRAINING_COMPLETED"
    assert completed["seed"] == SEED
    assert completed["condition"] == CONDITION
    assert completed["protocol_sha256"] == sha256(protocol_path)
    assert completed["training_csv_sha256"] == sha256(data_path)
    assert (model_dir / "model.safetensors").exists()
    assert completed["model_sha256"] == sha256(
        model_dir / "model.safetensors"
    )

    print("STAGE 13 — BASELINE SEED 42")
    print("Existing training run: VERIFIED")
    print("Optimizer updates:", completed["optimizer_updates"])
    print("Epoch losses:", completed["epoch_losses"])
    print("STATUS: BASELINE SEED 42 ALREADY COMPLETED")

else:
    # Remove the earlier smoke-test model to reclaim GPU memory.
    if "downstream_model" in globals():
        del downstream_model

    gc.collect()
    torch.cuda.empty_cache()

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    torch.backends.cudnn.benchmark = False

    tokenizer = AutoTokenizer.from_pretrained(
        cfg["model_id"],
        revision=cfg["model_revision_sha"],
        token=False,
        use_fast=True
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        cfg["model_id"],
        revision=cfg["model_revision_sha"],
        token=False,
        num_labels=4
    ).to(DEVICE)

    # Same parameter grouping will be used for both conditions.
    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")

    optimizer = torch.optim.AdamW(
        [
            {
                "params": [
                    p for n, p in model.named_parameters()
                    if p.requires_grad
                    and not any(x in n for x in no_decay)
                ],
                "weight_decay": cfg["training"]["weight_decay"]
            },
            {
                "params": [
                    p for n, p in model.named_parameters()
                    if p.requires_grad
                    and any(x in n for x in no_decay)
                ],
                "weight_decay": 0.0
            }
        ],
        lr=cfg["training"]["learning_rate"],
        foreach=False
    )

    def lr_multiplier(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)

        return max(
            0.0,
            (total_updates - step)
            / max(1, total_updates - warmup_steps)
        )

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lr_multiplier
    )

    start_epoch = 0
    start_batch = 0
    global_step = 0

    epoch_losses = []
    loss_sum = 0.0
    examples_seen = 0

    if checkpoint_path.exists():
        state = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True
        )

        assert state["seed"] == SEED
        assert state["condition"] == CONDITION
        assert state["protocol_sha256"] == sha256(protocol_path)
        assert state["training_csv_sha256"] == sha256(data_path)

        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])

        start_epoch = state["epoch"]
        start_batch = state["next_batch"]
        global_step = state["global_step"]
        epoch_losses = state["epoch_losses"]
        loss_sum = state["loss_sum"]
        examples_seen = state["examples_seen"]

        torch.set_rng_state(state["cpu_rng_state"])
        torch.cuda.set_rng_state_all(
            state["cuda_rng_states"]
        )

        print(
            "RESUMED:",
            f"epoch={start_epoch + 1}",
            f"batch={start_batch}",
            f"updates={global_step}"
        )

    def save_checkpoint(epoch, next_batch):
        state = {
            "seed": SEED,
            "condition": CONDITION,
            "protocol_sha256": sha256(protocol_path),
            "training_csv_sha256": sha256(data_path),
            "epoch": epoch,
            "next_batch": next_batch,
            "global_step": global_step,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "cpu_rng_state": torch.get_rng_state(),
            "cuda_rng_states": torch.cuda.get_rng_state_all(),
            "epoch_losses": epoch_losses,
            "loss_sum": loss_sum,
            "examples_seen": examples_seen
        }

        temporary = checkpoint_path.with_suffix(".pt.tmp")
        torch.save(state, temporary)
        os.replace(temporary, checkpoint_path)

    model.train()
    optimizer.zero_grad(set_to_none=True)

    print("STAGE 13 — BASELINE TRAINING STARTED")
    print("Condition:", CONDITION)
    print("Seed:", SEED)
    print("Training records:", len(data))
    print("Epochs:", epochs)
    print("Batches per epoch:", batches_per_epoch)
    print("Total optimizer updates:", total_updates)
    print("Warmup updates:", warmup_steps)
    print("GPU:", torch.cuda.get_device_name(0))
    print("Holdout accessed: NO")

    for epoch in range(start_epoch, epochs):
        permutation = np.random.default_rng(
            SEED + epoch
        ).permutation(len(data))

        batch_begin = start_batch if epoch == start_epoch else 0

        for batch_index in range(batch_begin, batches_per_epoch):
            start = batch_index * batch_size
            end = min(start + batch_size, len(data))

            positions = permutation[start:end]

            encoded = tokenizer(
                [texts[int(i)] for i in positions],
                padding=True,
                truncation=True,
                max_length=cfg["training"]["max_length"],
                return_tensors="pt"
            )

            encoded = {
                k: v.to(DEVICE)
                for k, v in encoded.items()
            }

            targets = torch.tensor(
                labels[positions],
                dtype=torch.long,
                device=DEVICE
            )

            outputs = model(
                **encoded,
                labels=targets
            )

            loss = outputs.loss
            assert torch.isfinite(loss).item()

            loss_sum += loss.item() * len(positions)
            examples_seen += len(positions)

            remaining_in_group = (
                batches_per_epoch
                - (batch_index // accumulation) * accumulation
            )

            group_size = min(accumulation, remaining_in_group)

            (loss / group_size).backward()

            del outputs, loss, encoded, targets

            update_due = (
                (batch_index + 1) % accumulation == 0
                or batch_index + 1 == batches_per_epoch
            )

            if update_due:
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                global_step += 1

                if global_step % 100 == 0:
                    print(
                        f"Epoch {epoch + 1}/{epochs} | "
                        f"Optimizer updates {global_step}/{total_updates}"
                    )

                    save_checkpoint(epoch, batch_index + 1)

        epoch_loss = loss_sum / examples_seen
        epoch_losses.append(round(float(epoch_loss), 6))

        print(
            f"EPOCH {epoch + 1} COMPLETED | "
            f"Mean training loss: {epoch_loss:.6f}"
        )

        loss_sum = 0.0
        examples_seen = 0

        save_checkpoint(epoch + 1, 0)

    assert global_step == total_updates
    assert len(epoch_losses) == epochs

    model.eval()
    model_dir.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(
        model_dir,
        safe_serialization=True
    )
    tokenizer.save_pretrained(model_dir)

    weight_path = model_dir / "model.safetensors"
    assert weight_path.is_file()

    summary = {
        "status": "TRAINING_COMPLETED",
        "condition": CONDITION,
        "seed": SEED,
        "records": len(data),
        "epochs": epochs,
        "optimizer_updates": global_step,
        "warmup_updates": warmup_steps,
        "epoch_losses": epoch_losses,
        "model_id": cfg["model_id"],
        "model_revision_sha": cfg["model_revision_sha"],
        "protocol_sha256": sha256(protocol_path),
        "training_csv_sha256": sha256(data_path),
        "model_sha256": sha256(weight_path),
        "precision": "FLOAT32",
        "gradient_clipping": "NONE",
        "batch_order": "NUMPY_DEFAULT_RNG_SEED_PLUS_EPOCH",
        "optimizer": "ADAMW_DECOUPLED_WEIGHT_DECAY",
        "weight_decay_exclusions": list(no_decay),
        "evaluation_performed": False,
        "holdout_accessed": False,
        "reference_labels_accessed": False
    }

    temporary = summary_path.with_suffix(".json.tmp")
    temporary.write_text(
        json.dumps(summary, indent=2) + "\n",
        encoding="utf-8"
    )
    os.replace(temporary, summary_path)

    assert summary_path.is_file()

    # The verified final model replaces the resumable checkpoint.
    if checkpoint_path.exists():
        checkpoint_path.unlink()

    print("\nSTAGE 13 — BASELINE SEED 42 COMPLETED")
    print("Training records:", len(data))
    print("Epoch losses:", epoch_losses)
    print("Optimizer updates:", global_step)
    print("Model:", model_dir)
    print("Summary:", summary_path)
    print("Training reference labels accessed: NO")
    print("Holdout accessed: NO")
    print("Evaluation performed: NO")
    print("\nSTATUS: BASELINE SEED 42 TRAINING PASSED")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


STAGE 13 — BASELINE TRAINING STARTED
Condition: baseline
Seed: 42
Training records: 13621
Epochs: 2
Batches per epoch: 852
Total optimizer updates: 852
Warmup updates: 52
GPU: Tesla T4
Holdout accessed: NO
Epoch 1/2 | Optimizer updates 100/852
Epoch 1/2 | Optimizer updates 200/852
Epoch 1/2 | Optimizer updates 300/852
Epoch 1/2 | Optimizer updates 400/852
EPOCH 1 COMPLETED | Mean training loss: 0.451491
Epoch 2/2 | Optimizer updates 500/852
Epoch 2/2 | Optimizer updates 600/852
Epoch 2/2 | Optimizer updates 700/852
Epoch 2/2 | Optimizer updates 800/852
EPOCH 2 COMPLETED | Mean training loss: 0.230441


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


STAGE 13 — BASELINE SEED 42 COMPLETED
Training records: 13621
Epoch losses: [0.451491, 0.230441]
Optimizer updates: 852
Model: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/training_runs/baseline_seed_42/final_model
Summary: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/training_runs/baseline_seed_42/training_summary_v1.json
Training reference labels accessed: NO
Holdout accessed: NO
Evaluation performed: NO

STATUS: BASELINE SEED 42 TRAINING PASSED


In [6]:

import os
import gc
import json
import math
import random
import shutil
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
ROOT = BASE / "outputs/s_slte_downstream_v1"

SEED = 42
CONDITION = "corrected"

PROTOCOL_PATH = ROOT / "downstream_training_protocol_v1.json"
DATA_PATH = ROOT / "corrected_training_v1.csv"
PAIRED_PATH = ROOT / "baseline_training_v1.csv"

RUN_DIR = ROOT / "training_runs" / f"{CONDITION}_seed_{SEED}"
CHECKPOINT_PATH = RUN_DIR / "training_checkpoint.pt"
MODEL_DIR = RUN_DIR / "final_model"
SUMMARY_PATH = RUN_DIR / "training_summary_v1.json"

LOCAL_DIR = Path("/content/s_slte_corrected_seed42")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)


def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024 * 1024), b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def run_corrected_training():

    assert PROTOCOL_PATH.is_file()
    assert torch.cuda.is_available(), "Enable T4 GPU."

    with PROTOCOL_PATH.open(encoding="utf-8") as f:
        cfg = json.load(f)

    assert cfg["model_id"] == "distilbert/distilbert-base-uncased"
    assert cfg["model_revision_sha"] == (
        "12040accade4e8a0f71eabdb258fecc2e7e948be"
    )
    assert SEED in cfg["seeds"]
    assert cfg["num_labels"] == 4

    settings = cfg["training"]

    assert settings["epochs"] == 2
    assert settings["train_batch_size"] == 16
    assert settings["gradient_accumulation_steps"] == 2
    assert settings["learning_rate"] == 2e-5
    assert settings["optimizer"] == "AdamW"
    assert settings["lr_scheduler"] == "linear"
    assert settings["early_stopping"] is False
    assert settings["drop_last"] is False

    protocol_hash = sha256_file(PROTOCOL_PATH)

    # Copy frozen inputs to local storage before training.
    local_data = LOCAL_DIR / DATA_PATH.name
    local_paired = LOCAL_DIR / PAIRED_PATH.name

    shutil.copyfile(DATA_PATH, local_data)
    shutil.copyfile(PAIRED_PATH, local_paired)

    data_hash = sha256_file(local_data)
    paired_hash = sha256_file(local_paired)

    assert data_hash == cfg["input_sha256"]["corrected_csv"]
    assert paired_hash == cfg["input_sha256"]["baseline_csv"]

    if SUMMARY_PATH.exists():
        with SUMMARY_PATH.open(encoding="utf-8") as f:
            completed = json.load(f)

        assert completed["status"] == "TRAINING_COMPLETED"
        assert completed["condition"] == CONDITION
        assert completed["seed"] == SEED
        assert completed["protocol_sha256"] == protocol_hash
        assert completed["training_csv_sha256"] == data_hash
        assert completed["optimizer_updates"] == 852

        weights = MODEL_DIR / "model.safetensors"
        assert weights.is_file()
        assert sha256_file(weights) == completed["model_sha256"]

        print("STAGE 13 — CORRECTED SEED 42")
        print("Existing completed run: VERIFIED")
        print("Epoch losses:", completed["epoch_losses"])
        print("Optimizer updates:", completed["optimizer_updates"])
        print("\nSTATUS: CORRECTED SEED 42 TRAINING PASSED")
        return

    data = pd.read_csv(local_data)
    paired = pd.read_csv(local_paired)

    assert len(data) == len(paired) == 13621
    assert data["record_id"].is_unique
    assert data["record_id"].equals(paired["record_id"])
    assert data["text"].equals(paired["text"])
    assert data["text"].notna().all()
    assert data["label"].isin(range(4)).all()
    assert paired["label"].isin(range(4)).all()

    changed = int(
        (data["label"] != paired["label"]).sum()
    )

    assert changed == 705

    texts = data["text"].astype(str).tolist()
    labels = data["label"].to_numpy(dtype=np.int64)

    batch_size = settings["train_batch_size"]
    accumulation = settings["gradient_accumulation_steps"]
    epochs = settings["epochs"]

    batches_per_epoch = math.ceil(len(data) / batch_size)
    updates_per_epoch = math.ceil(
        batches_per_epoch / accumulation
    )
    total_updates = updates_per_epoch * epochs

    warmup_steps = math.ceil(
        total_updates * settings["warmup_ratio"]
    )

    assert batches_per_epoch == 852
    assert total_updates == 852
    assert warmup_steps == 52

    device = torch.device("cuda:0")

    # The previous cell may still hold the interrupted model.
    old = globals()

    memory_names = [
        "model", "optimizer", "scheduler", "tokenizer",
        "global_step", "epoch", "batch_index",
        "epoch_losses", "loss_sum", "examples_seen",
        "labels", "texts"
    ]

    memory_available = (
        all(name in old for name in memory_names)
        and old.get("CONDITION") == CONDITION
        and old.get("SEED") == SEED
    )

    if memory_available:
        memory_available = (
            isinstance(old["model"], torch.nn.Module)
            and isinstance(
                old["optimizer"], torch.optim.Optimizer
            )
            and 0 < int(old["global_step"]) < total_updates
            and int(old["epoch"]) in (0, 1)
            and 0 <= int(old["batch_index"]) < batches_per_epoch
            and np.array_equal(old["labels"], labels)
            and old["texts"] == texts
            and old["model"].training
            and next(old["model"].parameters()).device.type == "cuda"
        )

    if memory_available:
        memory_available = all(
            parameter.grad is None
            for parameter in old["model"].parameters()
        )

    if memory_available:
        old_epoch = int(old["epoch"])
        old_next_batch = int(old["batch_index"]) + 1

        expected_step = (
            old_epoch * updates_per_epoch
            + math.ceil(old_next_batch / accumulation)
        )

        memory_available = (
            old_next_batch < batches_per_epoch
            and old_next_batch % accumulation == 0
            and int(old["global_step"]) == expected_step
            and len(old["epoch_losses"]) == old_epoch
        )

    recovery_mode = "FRESH"

    if memory_available:
        model = old["model"]
        optimizer = old["optimizer"]
        scheduler = old["scheduler"]
        tokenizer = old["tokenizer"]

        start_epoch = int(old["epoch"])
        start_batch = int(old["batch_index"]) + 1
        global_step = int(old["global_step"])

        epoch_losses = list(old["epoch_losses"])
        loss_sum = float(old["loss_sum"])
        examples_seen = int(old["examples_seen"])

        recovery_mode = "IN_MEMORY"

    else:
        # A new initialization uses the same frozen seed.
        gc.collect()
        torch.cuda.empty_cache()

        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.benchmark = False

        tokenizer = AutoTokenizer.from_pretrained(
            cfg["model_id"],
            revision=cfg["model_revision_sha"],
            token=False,
            use_fast=True
        )

        model = AutoModelForSequenceClassification.from_pretrained(
            cfg["model_id"],
            revision=cfg["model_revision_sha"],
            token=False,
            num_labels=4
        ).to(device)

        no_decay = (
            "bias", "LayerNorm.weight", "layer_norm.weight"
        )

        optimizer = torch.optim.AdamW(
            [
                {
                    "params": [
                        p for n, p in model.named_parameters()
                        if p.requires_grad
                        and not any(x in n for x in no_decay)
                    ],
                    "weight_decay": settings["weight_decay"]
                },
                {
                    "params": [
                        p for n, p in model.named_parameters()
                        if p.requires_grad
                        and any(x in n for x in no_decay)
                    ],
                    "weight_decay": 0.0
                }
            ],
            lr=settings["learning_rate"],
            foreach=False
        )

        def lr_multiplier(step):
            if step < warmup_steps:
                return step / max(1, warmup_steps)

            return max(
                0.0,
                (total_updates - step)
                / max(1, total_updates - warmup_steps)
            )

        scheduler = torch.optim.lr_scheduler.LambdaLR(
            optimizer,
            lr_lambda=lr_multiplier
        )

        start_epoch = 0
        start_batch = 0
        global_step = 0

        epoch_losses = []
        loss_sum = 0.0
        examples_seen = 0

        # Restore the last completed atomic checkpoint.
        if CHECKPOINT_PATH.exists():
            state = torch.load(
                CHECKPOINT_PATH,
                map_location="cpu",
                weights_only=True
            )

            assert state["condition"] == CONDITION
            assert state["seed"] == SEED
            assert state["protocol_sha256"] == protocol_hash
            assert state["training_csv_sha256"] == data_hash

            model.load_state_dict(state["model"])
            optimizer.load_state_dict(state["optimizer"])
            scheduler.load_state_dict(state["scheduler"])

            start_epoch = int(state["epoch"])
            start_batch = int(state["next_batch"])
            global_step = int(state["global_step"])

            epoch_losses = list(state["epoch_losses"])
            loss_sum = float(state["loss_sum"])
            examples_seen = int(state["examples_seen"])

            assert 0 <= start_epoch <= epochs
            assert 0 <= start_batch <= batches_per_epoch
            assert global_step <= total_updates

            torch.set_rng_state(state["cpu_rng_state"])
            torch.cuda.set_rng_state_all(
                state["cuda_rng_states"]
            )

            recovery_mode = "DISK_CHECKPOINT"

    # Preserve exact training order for the matched pair.
    model.train()
    optimizer.zero_grad(set_to_none=True)

    def save_checkpoint(next_epoch, next_batch):
        state = {
            "condition": CONDITION,
            "seed": SEED,
            "protocol_sha256": protocol_hash,
            "training_csv_sha256": data_hash,
            "epoch": next_epoch,
            "next_batch": next_batch,
            "global_step": global_step,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "cpu_rng_state": torch.get_rng_state(),
            "cuda_rng_states": torch.cuda.get_rng_state_all(),
            "epoch_losses": epoch_losses,
            "loss_sum": loss_sum,
            "examples_seen": examples_seen
        }

        # Serialize locally; synchronize to Drive atomically.
        local_temp = LOCAL_DIR / "training_checkpoint.pt"
        drive_temp = CHECKPOINT_PATH.with_suffix(".pt.tmp")

        torch.save(state, local_temp)
        shutil.copyfile(local_temp, drive_temp)
        os.replace(drive_temp, CHECKPOINT_PATH)

        local_temp.unlink(missing_ok=True)

    print("STAGE 13 — S-SLTE CORRECTED TRAINING")
    print("Condition:", CONDITION)
    print("Seed:", SEED)
    print("Recovery mode:", recovery_mode)
    print("Starting optimizer update:", global_step)
    print("Total optimizer updates:", total_updates)
    print("Training records:", len(data))
    print("Changed labels:", changed)
    print("GPU:", torch.cuda.get_device_name(0))
    print("Holdout accessed: NO")

    for epoch in range(start_epoch, epochs):
        permutation = np.random.default_rng(
            SEED + epoch
        ).permutation(len(data))

        first_batch = (
            start_batch if epoch == start_epoch else 0
        )

        for batch_index in range(
            first_batch, batches_per_epoch
        ):
            begin = batch_index * batch_size
            end = min(begin + batch_size, len(data))

            positions = permutation[begin:end]

            encoded = tokenizer(
                [texts[int(i)] for i in positions],
                padding=True,
                truncation=True,
                max_length=settings["max_length"],
                return_tensors="pt"
            )

            encoded = {
                key: value.to(device)
                for key, value in encoded.items()
            }

            targets = torch.tensor(
                labels[positions],
                dtype=torch.long,
                device=device
            )

            outputs = model(**encoded, labels=targets)
            loss = outputs.loss

            assert torch.isfinite(loss).item()

            loss_sum += loss.item() * len(positions)
            examples_seen += len(positions)

            remaining_group = (
                batches_per_epoch
                - (batch_index // accumulation) * accumulation
            )

            group_size = min(
                accumulation,
                remaining_group
            )

            (loss / group_size).backward()

            del outputs, loss, encoded, targets

            update_due = (
                (batch_index + 1) % accumulation == 0
                or batch_index + 1 == batches_per_epoch
            )

            if update_due:
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                global_step += 1

                if global_step % 100 == 0:
                    print(
                        f"Epoch {epoch + 1}/{epochs} | "
                        f"Optimizer updates: "
                        f"{global_step}/{total_updates}"
                    )

                    save_checkpoint(
                        epoch,
                        batch_index + 1
                    )

        epoch_loss = loss_sum / examples_seen
        epoch_losses.append(
            round(float(epoch_loss), 6)
        )

        print(
            f"EPOCH {epoch + 1} COMPLETED | "
            f"Mean training loss: {epoch_loss:.6f}"
        )

        loss_sum = 0.0
        examples_seen = 0

        save_checkpoint(epoch + 1, 0)

    assert global_step == total_updates
    assert len(epoch_losses) == epochs

    model.eval()
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(
        MODEL_DIR,
        safe_serialization=True
    )
    tokenizer.save_pretrained(MODEL_DIR)

    weights_path = MODEL_DIR / "model.safetensors"
    assert weights_path.is_file()

    no_decay = (
        "bias", "LayerNorm.weight", "layer_norm.weight"
    )

    summary = {
        "status": "TRAINING_COMPLETED",
        "condition": CONDITION,
        "seed": SEED,
        "records": len(data),
        "epochs": epochs,
        "optimizer_updates": global_step,
        "warmup_updates": warmup_steps,
        "epoch_losses": epoch_losses,
        "model_id": cfg["model_id"],
        "model_revision_sha": cfg["model_revision_sha"],
        "protocol_sha256": protocol_hash,
        "training_csv_sha256": data_hash,
        "model_sha256": sha256_file(weights_path),
        "precision": "FLOAT32",
        "gradient_clipping": "NONE",
        "batch_order": "NUMPY_DEFAULT_RNG_SEED_PLUS_EPOCH",
        "optimizer": "ADAMW_DECOUPLED_WEIGHT_DECAY",
        "weight_decay_exclusions": list(no_decay),
        "recovery_mode": recovery_mode,
        "evaluation_performed": False,
        "holdout_accessed": False,
        "reference_labels_accessed": False
    }

    temp_summary = SUMMARY_PATH.with_suffix(".json.tmp")

    temp_summary.write_text(
        json.dumps(summary, indent=2) + "\n",
        encoding="utf-8"
    )

    os.replace(temp_summary, SUMMARY_PATH)

    assert SUMMARY_PATH.is_file()

    # Delete the resumable file only after completion.
    CHECKPOINT_PATH.unlink(missing_ok=True)

    print("\nSTAGE 13 — CORRECTED SEED 42 COMPLETED")
    print("Recovery mode:", recovery_mode)
    print("Epoch losses:", epoch_losses)
    print("Optimizer updates:", global_step)
    print("Final model:", MODEL_DIR)
    print("Training reference labels accessed: NO")
    print("Holdout accessed: NO")
    print("Evaluation performed: NO")
    print("\nSTATUS: CORRECTED SEED 42 TRAINING PASSED")


run_corrected_training()


STAGE 13 — S-SLTE CORRECTED TRAINING
Condition: corrected
Seed: 42
Recovery mode: IN_MEMORY
Starting optimizer update: 700
Total optimizer updates: 852
Training records: 13621
Changed labels: 705
GPU: Tesla T4
Holdout accessed: NO
Epoch 2/2 | Optimizer updates: 800/852
EPOCH 2 COMPLETED | Mean training loss: 0.171706


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


STAGE 13 — CORRECTED SEED 42 COMPLETED
Recovery mode: IN_MEMORY
Epoch losses: [0.394695, 0.171706]
Optimizer updates: 852
Final model: /content/drive/MyDrive/S_SLTE_Thesis/outputs/s_slte_downstream_v1/training_runs/corrected_seed_42/final_model
Training reference labels accessed: NO
Holdout accessed: NO
Evaluation performed: NO

STATUS: CORRECTED SEED 42 TRAINING PASSED


In [8]:

import json
import hashlib
from pathlib import Path

from transformers import AutoConfig

# Stage 13 — Verify completed matched training pair.
BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
ROOT = BASE / "outputs/s_slte_downstream_v1"

PROTOCOL_PATH = ROOT / "downstream_training_protocol_v1.json"

with PROTOCOL_PATH.open(encoding="utf-8") as f:
    protocol = json.load(f)

def file_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

protocol_hash = file_sha256(PROTOCOL_PATH)

assert protocol["model_id"] == "distilbert/distilbert-base-uncased"
assert protocol["seeds"] == [42, 123, 2026]
assert protocol["training"]["epochs"] == 2

SEED = 42
summaries = {}

for condition in ("baseline", "corrected"):

    run_dir = (
        ROOT / "training_runs"
        / f"{condition}_seed_{SEED}"
    )

    summary_path = run_dir / "training_summary_v1.json"
    model_dir = run_dir / "final_model"
    weights_path = model_dir / "model.safetensors"
    config_path = model_dir / "config.json"

    assert summary_path.is_file()
    assert weights_path.is_file()
    assert config_path.is_file()

    with summary_path.open(encoding="utf-8") as f:
        summary = json.load(f)

    # Read the saved Hugging Face model configuration.
    model_config = AutoConfig.from_pretrained(
        str(model_dir),
        local_files_only=True
    )

    assert model_config.num_labels == 4
    assert model_config.model_type == "distilbert"

    assert summary["status"] == "TRAINING_COMPLETED"
    assert summary["condition"] == condition
    assert summary["seed"] == SEED
    assert summary["records"] == 13621
    assert summary["epochs"] == 2
    assert summary["optimizer_updates"] == 852
    assert summary["warmup_updates"] == 52

    assert summary["model_id"] == protocol["model_id"]
    assert summary["model_revision_sha"] == (
        protocol["model_revision_sha"]
    )

    assert summary["protocol_sha256"] == protocol_hash

    expected_data_hash = protocol["input_sha256"][
        f"{condition}_csv"
    ]

    assert summary["training_csv_sha256"] == (
        expected_data_hash
    )

    assert len(summary["model_sha256"]) == 64
    assert len(summary["epoch_losses"]) == 2
    assert all(
        isinstance(loss, (int, float)) and loss >= 0
        for loss in summary["epoch_losses"]
    )

    assert weights_path.stat().st_size > 100_000_000

    assert summary["evaluation_performed"] is False
    assert summary["holdout_accessed"] is False
    assert summary["reference_labels_accessed"] is False

    summaries[condition] = summary

baseline = summaries["baseline"]
corrected = summaries["corrected"]

# Verify matched experiment settings.
assert baseline["seed"] == corrected["seed"]
assert baseline["epochs"] == corrected["epochs"]
assert baseline["optimizer_updates"] == (
    corrected["optimizer_updates"]
)
assert baseline["protocol_sha256"] == (
    corrected["protocol_sha256"]
)
assert baseline["model_revision_sha"] == (
    corrected["model_revision_sha"]
)

assert baseline["epoch_losses"] == [
    0.451491, 0.230441
]

assert corrected["epoch_losses"] == [
    0.394695, 0.171706
]

print("STAGE 13 — MATCHED SEED 42 VERIFICATION")

print("\nBASELINE")
print("Epoch losses:", baseline["epoch_losses"])
print("Final model: FOUND")
print("Model configuration: VERIFIED")

print("\nS-SLTE CORRECTED")
print("Epoch losses:", corrected["epoch_losses"])
print("Final model: FOUND")
print("Model configuration: VERIFIED")
print(
    "Recovery mode:",
    corrected.get("recovery_mode", "NOT_RECORDED")
)

print("\nMATCHED TRAINING")
print("Same pretrained revision: YES")
print("Same random seed: YES")
print("Same frozen protocol: YES")
print("Same epochs: YES")
print("Same optimizer update count: YES")
print("Number of output classes: 4")
print("Model artifacts: FOUND")
print("Training summaries: VERIFIED")

print("\nRESEARCH INTEGRITY")
print("Training reference labels accessed: NO")
print("Holdout accessed: NO")
print("Evaluation performed: NO")
print("Model weights independently rehashed: NO")
print("Original artifacts modified: NO")

print("\nSTATUS: MATCHED SEED 42 ARTIFACTS VERIFIED")


STAGE 13 — MATCHED SEED 42 VERIFICATION

BASELINE
Epoch losses: [0.451491, 0.230441]
Final model: FOUND
Model configuration: VERIFIED

S-SLTE CORRECTED
Epoch losses: [0.394695, 0.171706]
Final model: FOUND
Model configuration: VERIFIED
Recovery mode: IN_MEMORY

MATCHED TRAINING
Same pretrained revision: YES
Same random seed: YES
Same frozen protocol: YES
Same epochs: YES
Same optimizer update count: YES
Number of output classes: 4
Model artifacts: FOUND
Training summaries: VERIFIED

RESEARCH INTEGRITY
Training reference labels accessed: NO
Holdout accessed: NO
Evaluation performed: NO
Model weights independently rehashed: NO
Original artifacts modified: NO

STATUS: MATCHED SEED 42 ARTIFACTS VERIFIED
